In [ ]:
%pip install -q ultralytics kagglehub
import random, shutil
from pathlib import Path
import matplotlib.pyplot as plt
import torch
import kagglehub
from ultralytics import YOLO
SEED, IMG, EPOCHS = 42, 224, 10
random.seed(SEED)

In [ ]:
# Download dataset
DATA = Path(kagglehub.dataset_download(
    "kritikseth/fruit-and-vegetable-image-recognition"
))
if (DATA / "validation").exists() and not (DATA / "val").exists():
    shutil.copytree(DATA / "validation", DATA / "val")

In [ ]:
# Check dataset
for split in ["train", "val", "test"]:
    folder = DATA / split
    if folder.exists():
        classes = sorted(p.name for p in folder.iterdir() if p.is_dir())
        count = sum(1 for p in folder.rglob("*") if p.suffix.lower() in
                    {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
        print(f"{split}: {len(classes)} classes, {count} images")
classes = sorted(p.name for p in (DATA / "train").iterdir() if p.is_dir())
print("Classes:", classes)

In [ ]:
# Display sample images
plt.figure(figsize=(14, 4))
for i, c in enumerate(random.sample(classes, min(8, len(classes)))):
    imgs = list((DATA / "train" / c).glob("*"))
    if imgs:
        plt.subplot(1, 8, i + 1)
        plt.imshow(plt.imread(imgs[0]))
        plt.title(c, fontsize=8)
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Train YOLOv8 classification model
model = YOLO("yolov8n-cls.pt")
model.train(
    data=str(DATA), epochs=EPOCHS, imgsz=IMG, batch=32,
    workers=0, device=0 if torch.cuda.is_available() else "cpu",
    optimizer="AdamW", lr0=0.001, seed=SEED,
    patience=5, plots=True, project="runs", name="veg_cls",
    exist_ok=True
)

In [ ]:
# Test model
save_dir = Path(model.trainer.save_dir)
metrics = model.val(data=str(DATA), split="test")
print(f"Test Top-1 Accuracy: {metrics.top1:.4f}")
print(f"Test Top-5 Accuracy: {metrics.top5:.4f}")

In [ ]:
# Display training results and confusion matrix
for name in ["results.png", "confusion_matrix.png"]:
    p = save_dir / name
    if p.exists():
        plt.figure(figsize=(10, 6))
        plt.imshow(plt.imread(p))
        plt.title(name)
        plt.axis("off")
        plt.show()

In [ ]:
# Test predictions
test_images = [p for p in (DATA / "test").rglob("*")
               if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
plt.figure(figsize=(14, 7))
for i, path in enumerate(random.sample(test_images, min(8, len(test_images)))):
    r = model.predict(str(path), imgsz=IMG, verbose=False)[0]
    predicted = r.names[r.probs.top1]
    actual = path.parent.name
    plt.subplot(2, 4, i + 1)
    plt.imshow(r.orig_img[..., ::-1])
    plt.title(f"Pred: {predicted} ({r.probs.top1conf:.2f})\nActual: {actual}",
              color="green" if predicted == actual else "red", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Save trained model
Path("outputs").mkdir(exist_ok=True)
shutil.copy2(save_dir / "weights" / "best.pt",
             "outputs/fruits_vegs_yolov8n_cls.pt")
print("Model saved successfully!")

In [ ]:
# Classify a new image
def classify_image(path, top_k=3):
    r = model.predict(str(path), imgsz=IMG, verbose=False)[0]
    scores = r.probs.data.cpu().numpy()
    plt.imshow(r.orig_img[..., ::-1])
    plt.title(f"Prediction: {r.names[r.probs.top1]}")
    plt.axis("off")
    plt.show()
    return {r.names[i]: round(float(scores[i]), 4)
            for i in r.probs.top5[:top_k]}
# Example:
# classify_image("fruit.jpg")